# Lezione 04 — Tracciamento degli esperimenti

**Corso:** MLOps e Monitoraggio con Python · **Blocco 2:** Versionare e misurare
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/04_seguimiento_experimentos_IT.md`

## Cosa vedrai in questa lezione

- Perché annotare gli esperimenti "a mano" (in un quaderno o nel nome del file) smette di funzionare dopo una manciata di prove.
- **MLflow Tracking** con un archivio di file **locale** (`mlflow.set_tracking_uri("file:" + ...)`, mai un server), dentro la cartella di lavoro effimera.
- Registrare davvero, con `mlflow.start_run()`, `log_param`, `log_metric` e `log_artifact`, **10 configurazioni reali** di un modello su `datos/churn.csv`.
- Leggere i run salvati con `mlflow.search_runs()` (un `DataFrame` di pandas) e confrontarli con un **criterio esplicito**: non solo la metrica principale, anche un indicatore di costo e la stabilità tra semi.
- Registrare il modello scelto come **artefatto versionato** con `mlflow.sklearn.log_model`, primo passo verso il registro dei modelli della lezione 8.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

## Prerequisiti

- `pandas`, `scikit-learn` e `mlflow` (la cella seguente installa solo ciò che manca, anche in Google Colab).
- La cartella `datos/` del corso, con `churn.csv` all'interno, e aver letto `datos/LEEME.md` (proporzione di abbandono, accuratezza ingannevole).
- Dalla lezione 3: l'idea di cartella di lavoro effimera e di "remoto"/archivio locale.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca.

In [ ]:
import os
import subprocess
import sys

# Fissiamo queste variabili PRIMA della prima volta che si importa mlflow (anche il controllo qui sotto),
# perché nessun messaggio automatico di MLflow dipenda dall'ora di sistema nell'output di questo notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("mlflow", "mlflow")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, definiamo `cargar_datos` e prepariamo le variabili d'ambiente di MLflow. Tutti gli addestramenti di questa lezione usano `SEMILLA_BASE = 42` (fissata più avanti), quindi non serve nessun altro seme aggiuntivo tranne quando confrontiamo esplicitamente più semi (sezione 7).

In [ ]:
import os
import shutil
import time

import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

# Variabili d'ambiente di MLflow: tolgono l'avviso dell'assistente, permettono l'archivio di file
# (in MLflow 3.x il backend a file è in modalità manutenzione e va accettato esplicitamente)
# e disattivano la barra di avanzamento nel salvare/caricare il modello (non serve in uno script e non è riproducibile in testo)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import mlflow.sklearn
import logging

# I messaggi INFO di MLflow stesso includono l'ora di sistema: li silenziamo perché
# l'output di questo notebook non dipenda dall'orologio del tuo computer
logging.getLogger("mlflow").setLevel(logging.ERROR)  # WARNING no basta: mlflow.utils.requirements_utils avisa en WARNING con detalles no deterministicos

print("Librerie e funzioni ausiliarie pronte. Versione di MLflow:", mlflow.__version__)

---
## 1. Perché un quaderno di appunti smette di funzionare

Simuliamo, con solo 3 righe, come si annotano a mano gli esperimenti in molti progetti reali all'inizio.

In [ ]:
# Così si annotano gli esperimenti "a mano" in un quaderno o nel nome del file
cuaderno_manual = [
    "modelo_v1.pkl -- regressione logistica, senza bilanciare, acc tipo 0.88 credo",
    "modelo_v2_buono.pkl -- ho provato class_weight balanced, meglio ma non ho annotato l'AUC",
    "modelo_v2_finale_DEFINITIVO.pkl -- di nuovo balanced, ho cambiato qualcosa di C, non ricordo quale valore",
]
for linea in cuaderno_manual:
    print("-", linea)

# Domanda concreta: quale dei tre ha l'AUC migliore? con quale C è stato addestrato il secondo?
# Risposta con questo "quaderno": non si può sapere senza indovinare o senza aprire ogni .pkl a mano.

**Cosa osservi?** Con 3 "esperimenti" è già impossibile rispondere con certezza a quale AUC ha avuto ciascuno o con quale `C` è stato addestrato il secondo. Con 10 configurazioni (quello che confronteremo in questa lezione) un quaderno così è direttamente ingestibile: non si può filtrare, ordinare né confrontare senza aprire ogni file a mano. È esattamente questo che risolve MLflow Tracking.

---
## 2. Cartella di lavoro effimera e archivio locale di MLflow

Come nella lezione 3: creiamo `proyecto_churn/` da zero, e al suo interno (mai in `datos/` né nella cartella del corso) l'archivio di file di MLflow.

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

In [ ]:
# L'archivio di MLflow è una cartella LOCALE dentro la cartella effimera: mai un server
RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)

EXPERIMENTO = "churn_experimentos"
mlflow.set_experiment(EXPERIMENTO)

print("Tracking URI configurato:", mlflow.get_tracking_uri())
print("Esperimento attivo:", EXPERIMENTO)
assert mlflow.get_tracking_uri().startswith("file:") and "proyecto_churn" in mlflow.get_tracking_uri()

**Cosa osservi?** `mlflow.get_tracking_uri()` inizia con `file:` e contiene `proyecto_churn`: è un archivio di file locale, dentro la cartella effimera di questa lezione, non un server né una cartella condivisa. Se cancelli `proyecto_churn/` (lo facciamo nell'ultima cella), la cronologia degli esperimenti sparisce con essa: è esattamente quello che vogliamo in un notebook ripetibile.

---
## 3. I dati e la linea di base (perché l'accuratezza da sola inganna)

Carichiamo `churn.csv`, dividiamo in addestramento/test con un unico seme fisso, e ricordiamo la proporzione di abbandono di `datos/LEEME.md`.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]

X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y = df["abandono"]

SEMILLA_BASE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA_BASE, stratify=y
)

proporcion_abandono = round(float(y.mean()), 4)
print("righe totali:", len(df))
print("proporzione di abbandono (dataset completo):", proporcion_abandono)
print("righe di addestramento / test:", len(X_train), "/", len(X_test))
assert len(X_train) + len(X_test) == len(df)

In [ ]:
def construir_preprocesador():
    """Stesso preprocessamento per le 10 configurazioni: OneHotEncoder sulle categoriche, scaling sulle numeriche."""
    return ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
        ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
    ])


NOMBRE_HIPERPARAMETRO = {"regresion_logistica": "C", "arbol_decision": "profundidad_max"}


def entrenar_pipeline(modelo, class_weight, valor_hp, semilla, X_ent, y_ent):
    """Addestra una Pipeline completa (preprocessamento + modello) con una configurazione concreta."""
    if modelo == "regresion_logistica":
        clasificador = LogisticRegression(class_weight=class_weight, random_state=semilla, max_iter=1000, C=valor_hp)
    else:
        clasificador = DecisionTreeClassifier(class_weight=class_weight, random_state=semilla, max_depth=valor_hp)
    pipeline = Pipeline([("preprocesado", construir_preprocesador()), ("clasificador", clasificador)])
    pipeline.fit(X_ent, y_ent)
    return pipeline, clasificador


def evaluar_pipeline(pipeline, clasificador, modelo, X_prueba, y_prueba):
    """Calcola le metriche e un indicatore di costo computazionale deterministico (non l'orologio di parete)."""
    predicciones = pipeline.predict(X_prueba)
    probabilidades = pipeline.predict_proba(X_prueba)[:, 1]
    exactitud = accuracy_score(y_prueba, predicciones)
    f1 = f1_score(y_prueba, predicciones, zero_division=0)
    auc = roc_auc_score(y_prueba, probabilidades)
    if modelo == "regresion_logistica":
        coste = int(clasificador.n_iter_[0])
    else:
        coste = int(clasificador.tree_.node_count)
    return predicciones, exactitud, f1, auc, coste

**Cosa osservi?** L'`abandono` è dell'11,7 % (coincide con `datos/LEEME.md`). Con una classe così minoritaria, un modello che predice **sempre** "non abbandona" indovina già circa l'88 % delle volte: la sezione seguente lo verifica addestrando davvero la configurazione più semplice.

---
## 4. Registrare un addestramento con MLflow, passo dopo passo

Addestriamo la configurazione 1 (regressione logistica, senza correggere lo sbilanciamento) e la registriamo con le quattro chiamate chiave di MLflow: `mlflow.start_run()`, `log_param`, `log_metric` e `log_artifact`. Come indicatore di costo computazionale usiamo il **numero di iterazioni** del solver (`n_iter_`): è deterministico (non dipende dalla velocità della tua macchina in questo momento), a differenza del tempo di orologio, che registriamo comunque (`tiempo_entrenamiento_seg`) ma non confrontiamo cifra per cifra in questo notebook.

In [ ]:
# Config 1: regressione logistica, senza correggere lo sbilanciamento (la linea di base "ingenua")
t0 = time.perf_counter()
pipeline_1, clasificador_1 = entrenar_pipeline("regresion_logistica", None, 1.0, SEMILLA_BASE, X_train, y_train)
tiempo_1 = time.perf_counter() - t0
predicciones_1, exactitud_1, f1_1, auc_1, coste_1 = evaluar_pipeline(pipeline_1, clasificador_1, "regresion_logistica", X_test, y_test)

with mlflow.start_run(run_name="config_1"):
    mlflow.set_tag("fase", "comparacion")
    # log_param: iperparametri e metadati di QUESTO addestramento concreto
    mlflow.log_param("config_id", 1)
    mlflow.log_param("modelo", "regresion_logistica")
    mlflow.log_param("class_weight", "ninguno")
    mlflow.log_param("hiperparametro", "C=1.0")
    mlflow.log_param("semilla", SEMILLA_BASE)
    # log_metric: risultati numerici di QUESTO addestramento concreto
    mlflow.log_metric("exactitud", exactitud_1)
    mlflow.log_metric("f1", f1_1)
    mlflow.log_metric("auc", auc_1)
    mlflow.log_metric("coste_computo", coste_1)
    mlflow.log_metric("tiempo_entrenamiento_seg", tiempo_1)
    # log_artifact: un file reale generato da questo addestramento (qui, un report di classificazione)
    informe_1 = classification_report(y_test, predicciones_1, zero_division=0)
    ruta_informe_1 = os.path.join(CARPETA, "informe_config_1.txt")
    with open(ruta_informe_1, "w") as f:
        f.write(informe_1)
    mlflow.log_artifact(ruta_informe_1)
    id_ejecucion_1 = mlflow.active_run().info.run_id

print("accuratezza:", round(exactitud_1, 4))
print("f1:", round(f1_1, 4))
print("auc:", round(auc_1, 4))
print("coste_computo (iterazioni del solver):", coste_1)
assert round(exactitud_1, 3) == round(float((y_test == 0).mean()), 3), "l'accuratezza dovrebbe coincidere con quella di predire sempre 'non abbandona'"

**Cosa osservi?** L'accuratezza (≈0,88) è praticamente uguale a quella di predire sempre "non abbandona" (l'`assert` lo verifica), ma `f1` è 0,0: il modello non identifica nemmeno un cliente che abbandona davvero. Guardando solo l'accuratezza, questa configurazione sembrerebbe buona; guardando l'f1, è inutile per l'obiettivo reale (trovare chi sta per abbandonare).

---
## 5. Confrontare 10 configurazioni reali

Definiamo le 10 configurazioni (5 di regressione logistica, 5 di albero di decisione, variando `class_weight` e l'iperparametro principale) e registriamo le altre 9 con la stessa funzione.

In [ ]:
# Le 10 configurazioni che confronteremo: 5 di regressione logistica, 5 di albero di decisione,
# variando class_weight e l'iperparametro principale di ogni modello
CONFIGURACIONES = [
    (1, "regresion_logistica", None, 1.0),
    (2, "regresion_logistica", None, 0.1),
    (3, "regresion_logistica", "balanced", 1.0),
    (4, "regresion_logistica", "balanced", 0.1),
    (5, "regresion_logistica", "balanced", 10.0),
    (6, "arbol_decision", None, 5),
    (7, "arbol_decision", None, 10),
    (8, "arbol_decision", "balanced", 5),
    (9, "arbol_decision", "balanced", 10),
    (10, "arbol_decision", "balanced", 3),
]


def registrar_configuracion(config_id, modelo, class_weight, valor_hp, semilla, X_ent, y_ent, X_prueba, y_prueba, fase="comparacion"):
    """Addestra, valuta e registra in MLflow una configurazione: un'unica funzione per le 10 + la stabilità."""
    t0 = time.perf_counter()
    pipeline, clasificador = entrenar_pipeline(modelo, class_weight, valor_hp, semilla, X_ent, y_ent)
    tiempo = time.perf_counter() - t0
    predicciones, exactitud, f1, auc, coste = evaluar_pipeline(pipeline, clasificador, modelo, X_prueba, y_prueba)
    with mlflow.start_run(run_name=f"config_{config_id}_s{semilla}"):
        mlflow.set_tag("fase", fase)
        mlflow.log_param("config_id", config_id)
        mlflow.log_param("modelo", modelo)
        mlflow.log_param("class_weight", class_weight if class_weight else "ninguno")
        mlflow.log_param("hiperparametro", f"{NOMBRE_HIPERPARAMETRO[modelo]}={valor_hp}")
        mlflow.log_param("semilla", semilla)
        mlflow.log_metric("exactitud", exactitud)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("coste_computo", coste)
        mlflow.log_metric("tiempo_entrenamiento_seg", tiempo)
        informe = classification_report(y_prueba, predicciones, zero_division=0)
        ruta_informe = os.path.join(CARPETA, f"informe_config_{config_id}_s{semilla}.txt")
        with open(ruta_informe, "w") as f:
            f.write(informe)
        mlflow.log_artifact(ruta_informe)
        id_ejecucion = mlflow.active_run().info.run_id
    return pipeline, exactitud, f1, auc, coste, id_ejecucion

In [ ]:
# Abbiamo già registrato la config 1 a mano; ora registriamo le altre 9 con la stessa funzione
pipelines = {1: pipeline_1}
for config_id, modelo, class_weight, valor_hp in CONFIGURACIONES[1:]:
    pipeline, exactitud, f1, auc, coste, _ = registrar_configuracion(
        config_id, modelo, class_weight, valor_hp, SEMILLA_BASE, X_train, y_train, X_test, y_test
    )
    pipelines[config_id] = pipeline

print("configurazioni registrate in totale:", len(pipelines))
assert len(pipelines) == 10

---
## 6. Leggere i run salvati con `mlflow.search_runs()`

`mlflow.search_runs()` restituisce un `DataFrame` di pandas con tutti i run dell'esperimento. Selezioniamo solo le colonne `params.*`/`metrics.*` che abbiamo registrato noi stessi.

In [ ]:
# mlflow.search_runs() restituisce un DataFrame di pandas con TUTTI i run dell'esperimento attivo
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])

# Ci teniamo SOLO i parametri e le metriche che abbiamo registrato noi stessi:
# non stampiamo mai start_time/end_time/run_id, che sono metadati automatici di MLflow e non sono riproducibili in testo
columnas_propias = [c for c in runs.columns if c.startswith("params.") or c.startswith("metrics.")]
columnas_propias = [c for c in columnas_propias if "tiempo_entrenamiento" not in c]

comparacion = runs[runs["tags.fase"] == "comparacion"][columnas_propias].copy()
comparacion["params.config_id"] = comparacion["params.config_id"].astype(int)
for columna in ["metrics.exactitud", "metrics.f1", "metrics.auc"]:
    comparacion[columna] = comparacion[columna].round(4)
comparacion = comparacion.sort_values("params.config_id").reset_index(drop=True)

print(comparacion[["params.config_id", "params.modelo", "params.class_weight", "params.hiperparametro",
                    "metrics.exactitud", "metrics.f1", "metrics.auc", "metrics.coste_computo"]].to_string(index=False))
assert len(comparacion) == 10

**Cosa osservi?** I 5 alberi di decisione hanno un AUC chiaramente peggiore (0,56-0,67) delle 5 regressioni logistiche (0,72-0,72): qui il modello lineare generalizza meglio dell'albero, anche con `class_weight="balanced"` in entrambi. Tra le regressioni logistiche, le due senza bilanciamento (config. 1 e 2) hanno l'AUC *più alto* delle dieci e, ciononostante, f1 = 0: se il criterio fosse "AUC più alto e basta", sceglieresti proprio il modello inutile della sezione 4.

---
## 7. Un criterio esplicito per scegliere

Non basta "la metrica principale più alta" (sezione 6). Applichiamo tre filtri, in ordine:

1. **AUC ≥ 0,70** (non dipende dalla soglia): scarta gli alberi di decisione.
2. Tra quelli rimasti, **F1 ≥ 0,30** (con soglia 0,5): scarta le due regressioni senza bilanciamento.
3. Tra i finalisti, **coste_computo minore** per primo (meno iterazioni del solver, addestramento più economico).

In [ ]:
# Criterio esplicito, in tre passi, NON solo "l'AUC più alto":
# 1) AUC >= 0.70 (scarta gli alberi, che qui generalizzano peggio)
# 2) tra quelli rimasti, F1 >= 0.30 (scarta le due regressioni senza bilanciamento, con f1 = 0)
# 3) tra i finalisti, coste_computo minore per primo (meno iterazioni del solver)
AUC_MINIMO = 0.70
F1_MINIMO = 0.30

registros = comparacion.to_dict("records")
supera_auc = [r for r in registros if r["metrics.auc"] >= AUC_MINIMO]
supera_f1 = [r for r in supera_auc if r["metrics.f1"] >= F1_MINIMO]
finalistas = sorted(supera_f1, key=lambda r: r["metrics.coste_computo"])

print("configurazioni che superano AUC >=", AUC_MINIMO, ":", [r["params.config_id"] for r in supera_auc])
print("...e inoltre F1 >=", F1_MINIMO, ":", [r["params.config_id"] for r in supera_f1])
print("finalisti ordinati per costo (minore prima):", [(r["params.config_id"], r["metrics.coste_computo"]) for r in finalistas])

assert {r["params.config_id"] for r in supera_f1} == {r["params.config_id"] for r in finalistas}
assert [r["params.config_id"] for r in supera_auc] == [1, 2, 3, 4, 5]
assert [r["params.config_id"] for r in supera_f1] == [3, 4, 5]

**Cosa osservi?** Tre configurazioni (3, 4 e 5) superano i due filtri: tutte e tre sono regressione logistica con `class_weight="balanced"`, e cambiano solo in `C`. L'AUC (0,7227-0,7232) e l'F1 (0,3085-0,3166) sono quasi identici tra loro; lo spareggio per costo dà già una prima preferenza (config. 3, con meno iterazioni), ma prima di decidere del tutto verifichiamo se questo risultato è stabile tra semi.

---
## 8. Stabilità tra semi

Un solo seme (42) può essere stato "fortunato" per una delle tre finaliste. Ripetiamo le tre con altri due semi (7 e 123, fissi perché il risultato sia riproducibile) e guardiamo la media e la deviazione di F1 e AUC.

In [ ]:
# Guardare solo il seme 42 può ingannare: ripetiamo i 3 finalisti con altri due semi
SEMILLAS_EXTRA = [7, 123]
IDS_FINALISTAS = [r["params.config_id"] for r in finalistas]
CONFIG_POR_ID = {c[0]: c for c in CONFIGURACIONES}

f1_por_config = {cid: [next(r["metrics.f1"] for r in registros if r["params.config_id"] == cid)] for cid in IDS_FINALISTAS}
auc_por_config = {cid: [next(r["metrics.auc"] for r in registros if r["params.config_id"] == cid)] for cid in IDS_FINALISTAS}

for semilla in SEMILLAS_EXTRA:
    X_ent_s, X_prueba_s, y_ent_s, y_prueba_s = train_test_split(X, y, test_size=0.25, random_state=semilla, stratify=y)
    for cid in IDS_FINALISTAS:
        _, modelo, class_weight, valor_hp = CONFIG_POR_ID[cid]
        _, exactitud, f1, auc, coste, _ = registrar_configuracion(
            cid, modelo, class_weight, valor_hp, semilla, X_ent_s, y_ent_s, X_prueba_s, y_prueba_s, fase="estabilidad"
        )
        f1_por_config[cid].append(f1)
        auc_por_config[cid].append(auc)

estabilidad = []
for cid in IDS_FINALISTAS:
    f1s = np.array(f1_por_config[cid])
    aucs = np.array(auc_por_config[cid])
    estabilidad.append((cid, round(f1s.mean(), 4), round(f1s.std(), 4), round(aucs.mean(), 4), round(aucs.std(), 4)))
    print(f"config {cid}: f1 media={round(f1s.mean(),4)} f1 desviacion={round(f1s.std(),4)}  auc media={round(aucs.mean(),4)} auc desviacion={round(aucs.std(),4)}")

# Verifichiamo che le esecuzioni di stabilità siano state salvate davvero in MLflow
runs_estabilidad = mlflow.search_runs(experiment_names=[EXPERIMENTO], filter_string="tags.fase = \'estabilidad\'")
print("esecuzioni di stabilità salvate in MLflow:", len(runs_estabilidad))
assert len(runs_estabilidad) == len(IDS_FINALISTAS) * len(SEMILLAS_EXTRA)

**Cosa osservi?** Con tre semi, l'F1 medio delle tre configurazioni resta praticamente lo stesso (≈0,31), ma la **deviazione** già le distingue: la config. 3 (`C=1.0`) è la più stabile (deviazione di F1 più bassa). Senza questo passo, avresti scelto solo in base al primo seme, senza sapere se il risultato regge.

---
## 9. Registrare il modello scelto come artefatto versionato

Addestriamo di nuovo la configurazione vincitrice (seme base) e la registriamo con `mlflow.sklearn.log_model`: il modello addestrato resta salvato come un artefatto in più di MLflow, versionato insieme ai suoi parametri e metriche. Questo non è ancora il registro dei modelli della lezione 8 (non ci sono versioni con nome né *stage*), ma è il pezzo di cui quel registro ha bisogno per esistere.

In [ ]:
# Il vincitore: minore deviazione di F1 tra semi (più stabile) E minor costo, con AUC e F1 già validati
ganador_id, f1_media, f1_std, auc_media, auc_std = min(estabilidad, key=lambda t: t[2])
_, modelo_ganador, class_weight_ganador, hp_ganador = CONFIG_POR_ID[ganador_id]

print("configurazione scelta:", ganador_id, modelo_ganador, class_weight_ganador, hp_ganador)
print("f1 media tra semi:", f1_media, "(deviazione:", f1_std, ")")

pipeline_ganador, clasificador_ganador = entrenar_pipeline(modelo_ganador, class_weight_ganador, hp_ganador, SEMILLA_BASE, X_train, y_train)
predicciones_ganador, exactitud_ganador, f1_ganador, auc_ganador, coste_ganador = evaluar_pipeline(
    pipeline_ganador, clasificador_ganador, modelo_ganador, X_test, y_test
)

with mlflow.start_run(run_name="modelo_elegido"):
    mlflow.set_tag("fase", "modelo_elegido")
    mlflow.log_param("config_id", ganador_id)
    mlflow.log_param("modelo", modelo_ganador)
    mlflow.log_param("class_weight", class_weight_ganador)
    mlflow.log_param("hiperparametro", f"{NOMBRE_HIPERPARAMETRO[modelo_ganador]}={hp_ganador}")
    mlflow.log_param("semilla", SEMILLA_BASE)
    mlflow.log_metric("exactitud", exactitud_ganador)
    mlflow.log_metric("f1", f1_ganador)
    mlflow.log_metric("auc", auc_ganador)
    resumen = (
        f"Configuracion elegida: {ganador_id} ({modelo_ganador}, class_weight={class_weight_ganador})\n"
        f"Criterio: AUC >= {AUC_MINIMO}, F1 >= {F1_MINIMO}, menor coste_computo, menor desviacion de F1 entre semillas.\n"
        f"f1 media entre semillas: {f1_media} (desviacion {f1_std})\n"
    )
    ruta_resumen = os.path.join(CARPETA, "resumen_modelo_elegido.txt")
    with open(ruta_resumen, "w") as f:
        f.write(resumen)
    mlflow.log_artifact(ruta_resumen)
    # log_model: registra il modello addestrato stesso come artefatto versionato (passo precedente al registro dei modelli della lezione 8)
    mlflow.sklearn.log_model(pipeline_ganador, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_ejecucion_ganadora = mlflow.active_run().info.run_id

print("modello salvato come artefatto di MLflow (dentro la cartella effimera).")

In [ ]:
# Verifica concreta: ricarichiamo il modello da MLflow e verifichiamo che predica UGUALE
modelo_recargado = mlflow.sklearn.load_model(f"runs:/{id_ejecucion_ganadora}/modelo")
predicciones_recargadas = modelo_recargado.predict(X_test)

print("le predizioni del modello ricaricato coincidono con quelle originali?", bool((predicciones_recargadas == predicciones_ganador).all()))
assert (predicciones_recargadas == predicciones_ganador).all()

**Cosa osservi?** Il modello ricaricato dall'archivio di MLflow (`mlflow.sklearn.load_model`) predice esattamente come l'originale: l'artefatto salvato è una copia fedele e funzionante del modello addestrato, non solo un registro delle sue metriche.

---
## 10. Pulizia finale

Eliminiamo `proyecto_churn/` (con il suo archivio di MLflow e tutti gli artefatti) perché non resti nessun residuo dentro il repository del corso, nemmeno ri-eseguendo questo notebook.

In [ ]:
# Pulizia finale: niente di questa lezione deve restare dentro il repository del corso
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo della lezione

- Annotare gli esperimenti "a mano" smette di funzionare non appena ce n'è più di una manciata: non si può filtrare né confrontare senza aprire ogni file.
- **MLflow Tracking**, con un archivio di file **locale** (mai un server in questo corso), registra parametri, metriche e artefatti di ogni addestramento reale con `start_run`/`log_param`/`log_metric`/`log_artifact`.
- Abbiamo confrontato **10 configurazioni reali** (regressione logistica e albero di decisione, con e senza `class_weight="balanced"`): l'AUC più alto (regressione senza bilanciamento) corrispondeva al modello con f1 = 0, la trappola esatta di guardare una sola metrica.
- Un **criterio esplicito** (soglia di AUC, soglia di F1, poi costo) e la **stabilità tra semi** sono necessari per non scegliere per caso.
- `mlflow.sklearn.log_model` registra il modello scelto come **artefatto versionato**: il primo passo verso il registro dei modelli della lezione 8.

*Continua con la lezione 5: Pipeline riproducibili.*